In [1]:
from google.colab import files
uploaded=files.upload()

Saving appointments.csv to appointments.csv
Saving bills.csv to bills.csv
Saving doctors.csv to doctors.csv
Saving hospital_logs.txt to hospital_logs.txt
Saving patients.csv to patients.csv
Saving patients_profile.json to patients_profile.json


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("HospitalAnalytics").getOrCreate()

patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
doctors = spark.read.csv("doctors.csv", header=True, inferSchema=True)
appointments = spark.read.csv("appointments.csv", header=True, inferSchema=True)
bills = spark.read.csv("bills.csv", header=True, inferSchema=True)
profiles = spark.read.json("patients_profile.json", multiLine=True)

In [5]:
patients.show()
doctors.show()
appointments.show()
bills.show()
profiles.show()
appointments.printSchema()
patients.count()
doctors.count()
appointments.show(5)

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+

+---------+-----------+--------------+---------+-------

In [6]:
patients.select("name","city","age").show()
doctors.select("doctor_name","specialization","consultation_fee").show()
patients.withColumnRenamed("name","patient_name").show()
doctors.withColumnRenamed("doctor_name","consultant_name").show()
patients.filter("city='Hyderabad'").show()
patients.filter("gender='Female'").show()
patients.filter("age > 35").show()
doctors.filter("city='Hyderabad'").show()
doctors.filter("specialization='Cardiology'").show()
doctors.filter("consultation_fee > 1000").show()

+------+---------+---+
|  name|     city|age|
+------+---------+---+
| Aarav|Hyderabad| 29|
| Priya|Bangalore| 34|
| Rahul|   Mumbai| 41|
| Sneha|    Delhi| 26|
| Kiran|  Chennai| 37|
| Meera|Hyderabad| 31|
|  Amit|     Pune| 45|
|  Neha|    Delhi| 28|
| Divya|Bangalore| 33|
|Vikram|   Mumbai| 52|
|Farhan|Hyderabad| 39|
|Simran|    Delhi| 25|
+------+---------+---+

+-----------+--------------+----------------+
|doctor_name|specialization|consultation_fee|
+-----------+--------------+----------------+
|  Dr Sharma|    Cardiology|            1200|
|    Dr Iyer|   Dermatology|             800|
|    Dr Khan|   Orthopedics|            1500|
|   Dr Reddy|    Pediatrics|             900|
|   Dr Mehta|     Neurology|            2000|
|    Dr Nair|    Cardiology|            1300|
|   Dr Verma|   Dermatology|             850|
|     Dr Rao|   Orthopedics|            1400|
+-----------+--------------+----------------+

+----------+------------+---------+---+------+-----------------+
|patient_id|p

In [7]:
patients.orderBy("age").show()
patients.orderBy("age", ascending=False).show()
patients.orderBy("age", ascending=False).show(5)
patients.orderBy("age").show(3)
doctors.orderBy("consultation_fee", ascending=False).show()
doctors.orderBy("consultation_fee", ascending=False).show(3)
doctors.orderBy("consultation_fee").show()
appointments.orderBy("appointment_date").show()
bills.orderBy("bill_amount", ascending=False).show()
bills.orderBy("bill_amount", ascending=False).show(5)

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
+----------+------+---------+---+------+-----------------+

+----------+------+---------+---+------+---------------

In [8]:
patients.groupBy("city").count().show()
patients.groupBy("gender").count().show()
doctors.groupBy("specialization").count().show()
patients.agg(avg("age")).show()
patients.agg(max("age")).show()
patients.agg(min("age")).show()
doctors.agg(avg("consultation_fee")).show()
doctors.agg(max("consultation_fee")).show()
bills.agg(sum("bill_amount")).show()
bills.agg(avg("bill_amount")).show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+

+------+-----+
|gender|count|
+------+-----+
|Female|    6|
|  Male|    6|
+------+-----+

+--------------+-----+
|specialization|count|
+--------------+-----+
|     Neurology|    1|
|   Dermatology|    2|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Orthopedics|    2|
+--------------+-----+

+--------+
|avg(age)|
+--------+
|    35.0|
+--------+

+--------+
|max(age)|
+--------+
|      52|
+--------+

+--------+
|min(age)|
+--------+
|      25|
+--------+

+---------------------+
|avg(consultation_fee)|
+---------------------+
|              1243.75|
+---------------------+

+---------------------+
|max(consultation_fee)|
+---------------------+
|                 2000|
+---------------------+

+----------------+
|sum(bill_amount)|
+----------------+
|           20250|
+----------------+

+-------------

In [9]:
patients.groupBy("city").count().show()
doctors.groupBy("city").count().show()
appointments.groupBy("status").count().show()
bills.groupBy("payment_status").sum("bill_amount").show()
bills.groupBy("payment_mode").sum("bill_amount").show()
bills.groupBy("payment_mode").avg("bill_amount").show()
appointments.groupBy("doctor_id").count().show()
appointments.groupBy("patient_id").count().show()
bills.groupBy("appointment_id").sum("bill_amount").show()
doctors.groupBy("specialization").avg("consultation_fee").show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|  Chennai|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    2|
+---------+-----+

+---------+-----+
|   status|count|
+---------+-----+
|Completed|   12|
|Cancelled|    2|
|  Pending|    2|
+---------+-----+

+--------------+----------------+
|payment_status|sum(bill_amount)|
+--------------+----------------+
|     Cancelled|            2050|
|       Pending|            2100|
|          Paid|           16100|
+--------------+----------------+

+------------+----------------+
|payment_mode|sum(bill_amount)|
+------------+----------------+
| Credit Card|            6300|
|        Cash|            4450|
|  Debit Card|            2600|
|         UPI|            6900|
+------------+----------------+

+------------+-

In [11]:
p_a = patients.join(appointments,"patient_id")
p_a.select("name","city","appointment_date","status").show()
d_a = doctors.join(appointments,"doctor_id")
d_a.select("doctor_name","specialization","appointment_date","status").show()
a_b = appointments.join(bills,"appointment_id")
a_b.select("appointment_id","status","bill_amount","payment_status").show()
pad = patients.join(appointments,"patient_id").join(doctors,"doctor_id")
pad.select("name","doctor_name","specialization","appointment_date").show()
full = patients.join(appointments,"patient_id").join(doctors,"doctor_id").join(bills,"appointment_id")
full.select("name","doctor_name","status","bill_amount","payment_mode").show()

+------+---------+----------------+---------+
|  name|     city|appointment_date|   status|
+------+---------+----------------+---------+
| Aarav|Hyderabad|      2024-03-01|Completed|
| Priya|Bangalore|      2024-03-01|Completed|
| Rahul|   Mumbai|      2024-03-02|Completed|
| Sneha|    Delhi|      2024-03-02|  Pending|
| Kiran|  Chennai|      2024-03-03|Completed|
| Meera|Hyderabad|      2024-03-03|Completed|
|  Amit|     Pune|      2024-03-04|Cancelled|
|  Neha|    Delhi|      2024-03-04|Completed|
| Divya|Bangalore|      2024-03-05|Completed|
|Vikram|   Mumbai|      2024-03-05|Completed|
|Farhan|Hyderabad|      2024-03-06|  Pending|
|Simran|    Delhi|      2024-03-06|Completed|
| Aarav|Hyderabad|      2024-03-07|Completed|
| Rahul|   Mumbai|      2024-03-07|Completed|
| Meera|Hyderabad|      2024-03-08|Cancelled|
| Divya|Bangalore|      2024-03-08|Completed|
+------+---------+----------------+---------+

+-----------+--------------+----------------+---------+
|doctor_name|specializa

In [12]:
patients.withColumn("age group",when(patients.age < 30,"Young") .when(patients.age < 40,"Adult").otherwise("Senior")).show()
patients.withColumn("hospital name", lit("BotCampus Hospital")).show()
doctors.withColumn("fee with tax", expr("consultation_fee * 1.18")).show()
bills.withColumn("bill with tax", expr("bill_amount * 1.18")).show()
bills.withColumn("bill in thousands", expr("bill_amount/1000")).show()
patients.withColumn("country", lit("India")).show()
doctors.withColumn("doctor label",expr("concat(doctor_name,' - ',specialization)")).show()
patients.withColumn("patient label",expr("concat(name,' - ',city)")).show()
bills.withColumn("high bill flag", expr("bill_amount > 1500")).show()
patients.withColumn("senior patient flag", expr("age > 40")).show()

+----------+------+---------+---+------+-----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|age group|
+----------+------+---------+---+------+-----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|    Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|    Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|   Senior|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|    Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|    Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|    Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   Senior|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|    Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|    Adult|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   Senior|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|    Adult|
|        12|Simran|    Delhi| 25|F

In [13]:
patients.withColumn("age_category",when(patients.age < 30,"Young").when(patients.age < 50,"Adult").otherwise("Senior")).show()
bills.withColumn("bill_category", when(bills.bill_amount >= 1500,"High").when(bills.bill_amount >= 1000,"Medium").otherwise("Low")).show()
doctors.withColumn("fee_category", when(doctors.consultation_fee >= 1500,"Premium") .when(doctors.consultation_fee >= 1000,"Standard").otherwise("Basic")).show()
appointments.withColumn("priority",when(appointments.status=="Pending","High").when(appointments.status=="Completed","Normal").otherwise("Low")).show()
patients.withColumn("zone",when(patients.city.isin("Hyderabad","Bangalore","Chennai"),"South").otherwise("Metro")).show()

+----------+------+---------+---+------+-----------------+------------+
|patient_id|  name|     city|age|gender|registration_date|age_category|
+----------+------+---------+---+------+-----------------+------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|       Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|       Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|       Adult|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|       Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|       Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|       Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|       Adult|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|       Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|       Adult|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|      Senior|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|      

In [14]:
patients = patients.withColumn("registration_date", to_date("registration_date"))
appointments = appointments.withColumn("appointment_date", to_date("appointment_date"))
patients.withColumn("year", year("registration_date")).show()
patients.withColumn("month", month("registration_date")).show()
appointments.withColumn("month", month("appointment_date")).show()
appointments.groupBy("appointment_date").count().show()
appointments.groupBy(month("appointment_date")).count().show()
patients.filter("registration_date > '2023-06-01'").show()
patients.withColumn("days_since_reg",datediff(current_date(),"registration_date")).show()
appointments.withColumn("days_since_appt", datediff(current_date(),"appointment_date")).show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|year|
+----------+------+---------+---+------+-----------------+----+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|2023|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|2023|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|2023|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|2023|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|2023|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|2023|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|2023|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|2023|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|2023|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|2023|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|2023|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|2023|
+----------+------+---------+---+------+

In [15]:
w_city = Window.partitionBy("city").orderBy(expr("age DESC"))
patients.withColumn("rank", rank().over(w_city)).show()
patients.withColumn("row_num", row_number().over(w_city)).filter("row_num=1").show()
w_spec = Window.partitionBy("specialization").orderBy(expr("consultation_fee DESC"))
doctors.withColumn("dense_rank", dense_rank().over(w_spec)).show()
doctors.withColumn("rank", rank().over(w_spec)).filter("rank=1").show()
patients.withColumn("rank", rank().over(w_city)).filter("rank<=2").show()
doctors.withColumn("rank", rank().over(Window.orderBy(expr("consultation_fee DESC")))).show()
patients.groupBy("city").count() .withColumn("rank", rank().over(Window.orderBy(expr("count DESC")))).show()
appointments.groupBy("doctor_id").count() .withColumn("rank", rank().over(Window.orderBy(expr("count DESC")))).show()
bills.withColumn("running_total",sum("bill_amount").over(Window.partitionBy("payment_mode").orderBy("bill_id"))).show()
appointments.withColumn("running_count",count("appointment_id").over(Window.partitionBy("doctor_id").orderBy("appointment_date"))).show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|rank|
+----------+------+---------+---+------+-----------------+----+
|         9| Divya|Bangalore| 33|Female|       2023-07-15|   1|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|   2|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|   1|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|   1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|   2|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|   3|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|   1|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|   2|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|   3|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|   1|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   2|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   1|
+----------+------+---------+---+------+

In [16]:
logs = spark.sparkContext.textFile("hospital_logs.txt")
logs.count()
logs.map(lambda x: x.split()[0]).collect()
logs.map(lambda x: x.split()[1]).collect()
logs.map(lambda x: x.split()[0]).distinct().collect()
logs.map(lambda x: (x.split()[1],1)) .reduceByKey(lambda a,b: a+b).collect()

[('login', 7), ('payment', 6), ('appointment', 6), ('logout', 1)]

In [17]:
profiles.printSchema()
profiles.selectExpr(
    "patient_id",
    "name",
    "contact.email as email",
    "contact.phone as phone"
).show()
profiles.selectExpr("patient_id","name","explode(allergies) as allergy").show()
profiles.selectExpr("explode(allergies) as allergy") .groupBy("allergy").count().show()
profiles.selectExpr("patient_id","name","explode(allergies) as allergy") .filter("allergy='Dust'").show()
profiles.selectExpr("explode(allergies) as allergy").distinct().show()
joined = profiles.join(patients,"patient_id")
joined.select("name","city","contact.email","allergies").show()
joined.selectExpr("city","explode(allergies) as allergy") .groupBy("city").count().show()

root
 |-- allergies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- name: string (nullable = true)
 |-- patient_id: long (nullable = true)

+----------+------+---------------+----------+
|patient_id|  name|          email|     phone|
+----------+------+---------------+----------+
|         1| Aarav| aarav@mail.com|9000011111|
|         2| Priya| priya@mail.com|9000022222|
|         3| Rahul| rahul@mail.com|9000033333|
|         6| Meera| meera@mail.com|9000066666|
|        10|Vikram|vikram@mail.com|9000101010|
+----------+------+---------------+----------+

+----------+------+-------+
|patient_id|  name|allergy|
+----------+------+-------+
|         1| Aarav|   Dust|
|         1| Aarav|Peanuts|
|         2| Priya| Pollen|
|         3| Rahul|   Dust|
|         3| Rahul|   Milk|
|         6| Meera|Seafood|
|        10|Vikram| Pollen|
|  

{"ts": "2026-04-27 10:19:58.711", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[AMBIGUOUS_REFERENCE] Reference `name` is ambiguous, could be: [`name`, `name`]. SQLSTATE: 42704", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor76.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "AMBIGUOUS_REFERENCE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o719.select.\n: org.apache.spark.sql.AnalysisException: [AMBIGUOUS_REFERENCE] Reference `name` is ambiguous, could be: [`name`, `name`]. SQLSTATE: 42704\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:2163)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:356)\n\tat org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:164)\n\tat org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpress

+-------+
|allergy|
+-------+
|   Milk|
|   Dust|
|Peanuts|
| Pollen|
|Seafood|
+-------+



AnalysisException: [AMBIGUOUS_REFERENCE] Reference `name` is ambiguous, could be: [`name`, `name`]. SQLSTATE: 42704

In [19]:
patients.createOrReplaceTempView("patients")
doctors.createOrReplaceTempView("doctors")
appointments.createOrReplaceTempView("appointments")
bills.createOrReplaceTempView("bills")
spark.sql("SELECT * FROM patients").show()
spark.sql("SELECT * FROM patients WHERE city='Hyderabad'").show()
spark.sql("SELECT city, COUNT(*) FROM patients GROUP BY city").show()
spark.sql("SELECT specialization, COUNT(*) FROM doctors GROUP BY specialization").show()
spark.sql("""
SELECT p.name, a.appointment_date
FROM patients p JOIN appointments a
ON p.patient_id = a.patient_id
""").show()

spark.sql("""
SELECT payment_mode, SUM(bill_amount)
FROM bills GROUP BY payment_mode
""").show()

spark.sql("""
SELECT doctor_name, consultation_fee,
RANK() OVER (ORDER BY consultation_fee DESC) as rank
FROM doctors
""").show()

spark.sql("""
SELECT p.name, SUM(b.bill_amount) as total
FROM patients p
JOIN appointments a ON p.patient_id=a.patient_id
JOIN bills b ON a.appointment_id=b.appointment_id
GROUP BY p.name
ORDER BY total DESC
LIMIT 1
""").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+

+----------+------+---------+---+------+---------------